## Objective
- Pad to make the image a square
- Resize to 224x224

In [1]:
import os
import cv2
import numpy as np
import config as cfg
from tqdm import tqdm

/home/takayuki/.local/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(
/home/takayuki/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
data_dir = cfg.PREPROCESSED_DIR
classes = cfg.CANONICAL_CLASSNAMES_LIST

target_dir = cfg.SQUARED_DIR
os.makedirs(target_dir, exist_ok=True)

In [3]:
total_count = 0

for species in tqdm(classes, desc="Processing species", unit="species", leave=False):
    class_count = 0
    species_path = os.path.join(data_dir, species)
    target_species_path = os.path.join(target_dir, species)
    os.makedirs(target_species_path, exist_ok=True)
    
    image_list = os.listdir(species_path)
    
    max_species_length = max(len(species) for species in classes)
    species = species.ljust(max_species_length)
    
    for image_name in tqdm(image_list, desc=f"Processing {species}", unit="img"):
        image_path = os.path.join(species_path, image_name)
        image = cv2.imread(image_path)
        
        if image is None:
            print(f"Warning: Unable to read image {image_path}. Skipping.")
            continue
        
        height, width = image.shape[:2]
        
        if height != width:
            
            if cfg.squaring_logic == 'white':
                size = max(height, width)
                squared_image = np.zeros((size, size, 3), dtype=image.dtype) + 255
                squared_image[:height, :width] = image
            
            elif cfg.squaring_logic == 'reflect':
                max_dim = max(height, width)
                
                pad_h = max_dim - height
                pad_w = max_dim - width
                
                top = pad_h // 2
                bottom = pad_h - top 
                
                left = pad_w // 2
                right = pad_w - left
                
                squared_image = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_REFLECT_101)
            
        else:
            squared_image = image
                
        target_image_path = os.path.join(target_species_path, image_name)
        cv2.imwrite(target_image_path, squared_image)
        
        
        class_count += 1
    
    total_count += class_count
    # print(f"Processed {class_count} images for species {species}.")

print(f"Total processed images: {total_count}")


Processing species:   0%|          | 0/21 [00:00<?, ?species/s]

Processing Vallisneria americana   : 100%|██████████| 10/10 [00:05<00:00,  1.92img/s]
                                                                        

Total processed images: 230
